# Identifying good reviews

Let's create a model to identify helpful reviews that people vote as helpful. They could be low rating reviews,

## Load Data

In [1]:
from pathlib import Path
from evalforge.utils import *

dataset_path = Path("data/clothes_review_10k.jsonl")
data = load_jsonl(dataset_path)

print(f"Number of examples: {len(data)}")
print(f"Number of reviews: {sum(len(example['reviews']) for example in data)}")
print("-"*100)
pprint(data[0])

Number of examples: 9999
Number of reviews: 159447
----------------------------------------------------------------------------------------------------
{
    "parent_asin": "5781728791",
    "main_category": "AMAZON FASHION",
    "title": "Women's Crewneck Striped Shirt Loose Colorblock Sweatshirt Pullover Top",
    "description": [],
    "average_rating": 4.0,
    "rating_number": 27,
    "asin": "5781728791",
    "features": [
        "Denim",
        "Hand Wash Only",
        "Imported",
        "Features: Crewneck, Long Sleeve, Stripe printed,Casual and basic Shirts for Women",
        "Casual pullover tops, perfect to pair with jeans, leggings,denim shorts",
        "This womens everyday loose striped shirt is perfect for Daily Wear, Party, School, Vacation, Office, Work, Home, Club, Night Out. Also a great choice as a gift for your wife, girlfriend, mom, daughter or sisters.",
        "Hand wash or Machine wash:Recommended with cold water"
    ],
    "price": "None",
    "images"

## LLM

In [2]:
import openai
import instructor
from pydantic import BaseModel, Field

client = instructor.from_openai(openai.OpenAI())

system_prompt = """You are an expert at evaluating the quality of clothing reviews. You are given a review of a clothing item.
Your task is to determine if the review is helpful and provides useful information about the clothing item."""

prompt_template = """The item to review is:

## Item Name
{title}

## Item Description
{description}

## Average Rating
{average_rating}

## Features
{features}

## Review

Rating: {review_rating}
Title: {review_title}

{review_content}

Is this review helpful and provides useful information about the clothing item and mark it as helpful or not helpful. Return in Json format.
"""


class ReviewEvaluation(BaseModel):
    helpful: bool = Field(description="Is the review helpful and provides useful information about the clothing item?")
    reason: str = Field(description="Reason for the evaluation")


In [3]:
example = data[0]

In [4]:
def format_example(example, review):
    return prompt_template.format(
        title=example["title"],
        description=listify(example["description"]),
        features=listify(example["features"]),
        average_rating=example["average_rating"],
        review_rating=review["rating"],
        review_title=review["title"],
        review_content=review["text"],
    )

In [5]:
print(format_example(example, example["reviews"][0]))

The item to review is:

## Item Name
Women's Crewneck Striped Shirt Loose Colorblock Sweatshirt Pullover Top

## Item Description
- None

## Average Rating
4.0

## Features
- Denim
- Hand Wash Only
- Imported
- Features: Crewneck, Long Sleeve, Stripe printed,Casual and basic Shirts for Women
- Casual pullover tops, perfect to pair with jeans, leggings,denim shorts
- This womens everyday loose striped shirt is perfect for Daily Wear, Party, School, Vacation, Office, Work, Home, Club, Night Out. Also a great choice as a gift for your wife, girlfriend, mom, daughter or sisters.
- Hand wash or Machine wash:Recommended with cold water

## Review

Rating: 4.0
Title: Soft and light

Bought for a Waldo costume. A little more burgundy than bright red. Material feels a little cheap, it is dyed red so you can see the white through if you stretch it. The inside is a light fleece feeling.

Is this review helpful and provides useful information about the clothing item and mark it as helpful or not h

In [6]:
def evaluate(example, review):
    return client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "system", "content": system_prompt}, 
                  {"role": "user", "content": format_example(example, review)}],
        response_model=ReviewEvaluation,
    )

evaluate(example, example["reviews"][0])


ReviewEvaluation(helpful=True, reason="The review provides specific details about the color ('a little more burgundy than bright red') and the material ('feels a little cheap', 'dyed red so you can see the white through if you stretch it', 'inside is a light fleece feeling'). These insights help potential buyers understand the item's look and feel, especially concerning color accuracy and material quality.")

## Helpful reviews

Let's look at the helpful reviews

In [9]:
HELPFUL_VOTE_THRESHOLD = 10

In [10]:
helpful_reviews = [(example, review) for example in data for review in example["reviews"] if review["helpful_vote"] > HELPFUL_VOTE_THRESHOLD]
print(f"Number of helpful reviews: {len(helpful_reviews)}")

Number of helpful reviews: 1776


In [11]:
product_info, review = helpful_reviews[0]
print(format_example(product_info, review))

The item to review is:

## Item Name
Corcoran Men's 10 Inch Side Zipper Jump Boot-M

## Item Description
- 10" Side Zipper Jump boot with Jump Boot Outsole , - PORON Shock Absorbing Insole , - Stitched Tap Sole , - NonTrip Beveled Heel , - Stay-Put Zipper and Zipper Flap Guard

## Average Rating
4.6

## Features
- 100% Leather
- Made in US
- Man Made sole
- Performance spit shineable leather
- Unlined
- Heavy duty YKK zipper with flap
- Poron cushioned insole
- Garrison army munson last for superior fit

## Review

Rating: 5.0
Title: that I don't have to wear a brace on my good leg.

Have been wearing Corcoran's since I was in the Army Airborne training, it take a few days to break them in, but that's understood,  because they are well made, with strong leather.  There support is the most important thing, because I lost a leg in Vietnam, wearing these boots give me enough support, that I don't have to wear a brace on my good leg.

Is this review helpful and provides useful information 

In [13]:
helpful_reviews_low_rating = [(example, review) for example in data for review in example["reviews"] if review["helpful_vote"] > HELPFUL_VOTE_THRESHOLD and review["rating"] < 3]
print(f"Number of helpful reviews with low rating: {len(helpful_reviews_low_rating)}")

Number of helpful reviews with low rating: 266


In [14]:
product_info, review = helpful_reviews_low_rating[0]
print(format_example(product_info, review))

The item to review is:

## Item Name
Casio Men's DBC150-1 Databank Digital Watch

## Item Description
- Mens Black Resin Case 150 Pg Telememo/150 Pg Sched.memo World Time, Alarm Chronograph 8 Digit Calculator with Phosphorescent Keypad Black Resin Band Scratch Resistant Mineral Crystal Japanese Quartz Movement Easy To Read LCD Display Electro-Luminescent Backlight Water Resistant
- Bringing you precision at a glance, the quartz-powered Casio Men's Databank Digital Watch #DBC150-1 features a blue-tone digital dial face, which is protected by a durable mineral dial window. An auto-calendar displays the date and month. It also includes a 150-page databank, an 8-digit calculator, a daily alarm, and a stopwatch function. To ensure easy wear, the deep-gray resin band is accompanied by a sturdy buckle clasp, and both the 35-millimeter case and stationary deep gray bezel are made of high-quality resin. Presenting an unsurpassed functionality, this innovative timepiece is designed to accommodat

In [15]:
evaluate(product_info, review)

ReviewEvaluation(helpful=False, reason="The review mainly focuses on the price hike and the perceived low-quality materials of the watch, such as the durability of the plastic, buttons, and straps. It does not offer detailed information on the watch's functionalities or how it performs over time with respect to those functionalities. Additionally, the review dwells on personal grievances about pricing rather than providing insights into the watch's core features or value compared to similar products. It does not aid potential buyers looking for information on the watch's specifications, features, or performance.")

this is hard...